# PolySight: verificación del checkpoint final en Google Colab

Este notebook verifica el checkpoint auditado `main16-baseline-seed42` mediante el código versionado de PolySight. No entrena, no modifica el modelo y no constituye validación clínica. Ejecuta las celdas en orden con un runtime de Python 3.11.

## 1. Configuración

Sustituye `REPO_URL` si el repositorio usa otra organización. Para un repositorio privado, configura el acceso mediante los secretos de Colab; no escribas tokens en el notebook. `GIT_REF` identifica el release reproducido.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/ORGANIZACION/polysight.git"
GIT_REF = "v0.1.0"
WORKSPACE = Path("/content/polysight-inference")
EXPECTED_CHECKPOINT_SHA256 = (
    "74aae659c028fc58a368f5a3f61a4c7875d1608a2cade0ade6da1ca5ebdb609c"
)

## 2. Verificar el runtime

PolySight 0.1.0 fija Python 3.11. El notebook se detiene si Colab expone otra versión, en lugar de cambiar silenciosamente el entorno experimental.

In [ ]:
import platform
import sys

print("Python:", platform.python_version())
print("Ejecutable:", sys.executable)
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("Este release requiere un runtime de Python 3.11")

## 3. Obtener e instalar el código exacto

El notebook instala el paquete; no copia las funciones de `src/polysight` a celdas independientes.

In [ ]:
import subprocess

def run(command: list[str], cwd: Path | None = None) -> None:
    print("+", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True)

if not WORKSPACE.exists():
    run(["git", "clone", REPO_URL, str(WORKSPACE)])
run(["git", "checkout", GIT_REF], cwd=WORKSPACE)
run([sys.executable, "-m", "pip", "install", "-e", "."], cwd=WORKSPACE)
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=WORKSPACE, text=True
).strip()
print("Commit:", commit)

In [ ]:
import mlflow
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("MLflow:", mlflow.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Cargar y verificar los archivos

Sube `best.pt` del run `cb29daac69dd4c9aa8a31ca621d08613` y una imagen endoscópica. Los datos clínicos o identificables no deben subirse sin autorización.

In [ ]:
from google.colab import files

print("Selecciona primero best.pt y después una imagen de prueba")
uploaded = files.upload()
uploaded_paths = [Path(name) for name in uploaded]
checkpoint_candidates = [path for path in uploaded_paths if path.suffix == ".pt"]
image_candidates = [
    path for path in uploaded_paths if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
]
if len(checkpoint_candidates) != 1 or len(image_candidates) != 1:
    raise ValueError("Debes subir exactamente un checkpoint .pt y una imagen")
CHECKPOINT_PATH = checkpoint_candidates[0]
IMAGE_PATH = image_candidates[0]

In [ ]:
import hashlib

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

checkpoint_sha256 = sha256_file(CHECKPOINT_PATH)
print("Checkpoint SHA-256:", checkpoint_sha256)
if checkpoint_sha256 != EXPECTED_CHECKPOINT_SHA256:
    raise ValueError("El checkpoint no coincide con main16-baseline-seed42")

## 5. Ejecutar inferencia

La función importada aplica exactamente la transformación `test`, reconstruye EfficientNet-B0 y devuelve el top-3.

In [ ]:
import json
from PIL import Image
from IPython.display import display
from polysight.predict import predict

display(Image.open(IMAGE_PATH).convert("RGB"))
predictions = predict(CHECKPOINT_PATH, IMAGE_PATH)
print(json.dumps(predictions, indent=2, ensure_ascii=False))

## Interpretación

Una ejecución correcta demuestra que el checkpoint conserva su identidad y que el pipeline puede producir inferencias en otro entorno. No demuestra que una predicción individual sea clínicamente correcta ni que Colab reproduzca bit a bit el entrenamiento de CEDIA.